# Dataset and Dataloader class

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Raw data

In [2]:
features = [
    [80, 1],   # index 0
    [25, 3],   # index 1
    [60, 0],   # index 2
    [30, 2],   # index 3
    [95, 1],   # index 4
    [40, 4],   # index 5
]
labels = [1, 0, 1, 0, 1, 0]

# Writing the Dataset Class

In [4]:
# Create our own class that inherits from Pytorch's "Dataset". Inheriting tells Pytorch that this class follows the Dataset rules, so the DataLoader knows how to use it.
class LoanDataset(Dataset):

    # This runs once when you create the dataset. Its job is to store the data, or for big datasets, store where the data is like file paths
    def __init__(self, features, labels):
        # Converts the python list into a tensor of shape (6, 2) and saves it inside the object
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    # Its job is to report the size. You never call __len__ yourself. When you write len(dataset), python calls it for you
    def __len__(self):
        # Returns the total number of rows, which is 6  in this case
        return len(self.features)

    # Its job is to return one sample.
    def __getitem__(self, idx):
        # This returns the row idx of the features and position idx of the labels. For idx = 3, that’s features [30., 2.] and label 0..
        return self.features[idx], self.labels[idx]

# Testing the Dataset by itself

In [5]:
# Always check your Dataset before adding the DataLoader. If something’s wrong here, it’s much easier to see.

dataset = LoanDataset(features, labels)
print(len(dataset))

6


# Adding the DataLoader

In [7]:
loader = DataLoader(dataset, batch_size = 2, shuffle = True)

for batch_num, (x_batch, y_batch) in enumerate(loader):
    print(f"Batch {batch_num + 1}")
    print(x_batch)
    print(y_batch)
    print("-" * 20)

Batch 1
tensor([[60.,  0.],
        [40.,  4.]])
tensor([1., 0.])
--------------------
Batch 2
tensor([[95.,  1.],
        [80.,  1.]])
tensor([1., 1.])
--------------------
Batch 3
tensor([[30.,  2.],
        [25.,  3.]])
tensor([0., 0.])
--------------------


# Using them in a training loop

In [8]:
# This is where Dataset and DataLoader connect to the nn parts we covered before.

model = nn.Linear(2, 1)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [9]:
for epoch in range(10):
    for x_batch, y_batch in loader:
        prediction = model(x_batch).squeeze(1)
        loss = loss_function(prediction, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # loss is a tensor containing one number. .item() takes that number out as a normal Python number, so it can be printed.
    print(f"Epoch {epoch + 1}, last batch loss: {loss.item():.4f}")

Epoch 1, last batch loss: 7.9161
Epoch 2, last batch loss: 2.9073
Epoch 3, last batch loss: 16.9992
Epoch 4, last batch loss: 0.8874
Epoch 5, last batch loss: 2.0419
Epoch 6, last batch loss: 3.3633
Epoch 7, last batch loss: 2.9306
Epoch 8, last batch loss: 7.8822
Epoch 9, last batch loss: 1.8814
Epoch 10, last batch loss: 1.0132
